In [1]:
!pip install transformers datasets evaluate


import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from datasets import Dataset
from transformers import XLMRobertaTokenizerFast, XLMRobertaForSequenceClassification, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import Trainer
from collections import Counter
from google.colab import files

# Ensure GPU is used if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Predefined list of German stopwords
german_stopwords = { ... }  # Use the predefined German stopwords from the original code

# Preprocess the text
def preprocess_text(text):
    text = re.sub(r'\S+@\S+\.\S+', '', text)  # Remove emails
    text = re.sub(r'[^A-Za-zäöüß ]+', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = ' '.join([word for word in text.split() if word not in german_stopwords])  # Remove stopwords
    return text

# Upload the dataset
uploaded = files.upload()
dataset_file_path = list(uploaded.keys())[0]
dataset = pd.read_csv(dataset_file_path)

# Check label distribution
label_counts = Counter(dataset['label'])
print("Original Label Distribution:", label_counts)

# Apply preprocessing
dataset["text"] = dataset["text"].apply(preprocess_text)

# Remove short and long texts (optional based on dataset quality)
dataset = dataset[dataset["text"].str.len() > 5]

# **Address Class Imbalance: Oversampling**
# Separate majority and minority classes
majority_class = dataset[dataset["label"] == dataset["label"].mode()[0]]
minority_class = dataset[dataset["label"] != dataset["label"].mode()[0]]

# Oversample the minority class
minority_upsampled = resample(
    minority_class,
    replace=True,  # Sample with replacement
    n_samples=len(majority_class),  # Match the majority class size
    random_state=42
)

# Combine oversampled minority class with the majority class
balanced_dataset = pd.concat([majority_class, minority_upsampled])

# Shuffle the dataset
balanced_dataset = balanced_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Check new label distribution
balanced_label_counts = Counter(balanced_dataset['label'])
print("Balanced Label Distribution:", balanced_label_counts)

# Split the dataset
train_df, test_df = train_test_split(balanced_dataset, test_size=0.2, random_state=42, stratify=balanced_dataset["label"])
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenizer and model setup
model_name = "xlm-roberta-base"  # Change to "xlm-roberta-large" if resources allow
tokenizer = XLMRobertaTokenizerFast.from_pretrained(model_name)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Tokenize datasets
train_tokenized_dataset = train_dataset.map(tokenize_function, batched=True)
test_tokenized_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns
train_tokenized_dataset = train_tokenized_dataset.remove_columns(["text", "__index_level_0__"])
test_tokenized_dataset = test_tokenized_dataset.remove_columns(["text", "__index_level_0__"])

# Rename labels
train_tokenized_dataset = train_tokenized_dataset.rename_column("label", "labels")
test_tokenized_dataset = test_tokenized_dataset.rename_column("label", "labels")

# Set format for PyTorch
train_tokenized_dataset.set_format("torch")
test_tokenized_dataset.set_format("torch")

# Define metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

# Define optimizer explicitly
def build_optimizer(model, learning_rate=1e-5, weight_decay=0.1):
    return AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Custom Trainer to integrate ReduceLROnPlateau
class CustomTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Ensure optimizer is explicitly defined
        if self.optimizer is None:
            self.optimizer = build_optimizer(self.model, learning_rate=self.args.learning_rate, weight_decay=self.args.weight_decay)
        # Define ReduceLROnPlateau scheduler
        self.reduce_lr_scheduler = ReduceLROnPlateau(
            self.optimizer,
            mode="min",  # Minimize the validation loss
            factor=0.5,  # Reduce learning rate by half
            patience=2,  # Number of epochs to wait before reducing LR
            threshold=0.01,  # Minimum improvement to be considered
            verbose=True,
        )

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        # Call the original evaluation method
        eval_metrics = super().evaluate(eval_dataset, ignore_keys, metric_key_prefix)

        # Update ReduceLROnPlateau scheduler
        val_loss = eval_metrics[f"{metric_key_prefix}_loss"]
        self.reduce_lr_scheduler.step(val_loss)

        return eval_metrics

# Training arguments
training_args = TrainingArguments(
    output_dir="./resultsMain",
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    weight_decay=0.1,
    warmup_steps=500,
    logging_steps=10,
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_accumulation_steps=2,
    max_grad_norm=1.0,
    report_to=[],  # Disable external reporting
)

# Model initialization
model = XLMRobertaForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# Define optimizer
optimizer = build_optimizer(model, learning_rate=training_args.learning_rate, weight_decay=training_args.weight_decay)

# Trainer setup with explicit optimizer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized_dataset,
    eval_dataset=test_tokenized_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # Early stopping
    optimizers=(optimizer, None)  # Pass explicit optimizer, scheduler handled in CustomTrainer
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print("Final Evaluation Results:")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1-Score: {eval_results['eval_f1']:.4f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 890.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
Using device: cuda


Saving gahd_filtered.csv to gahd_filtered.csv
Original Label Distribution: Counter({0: 6330, 1: 4666})
Balanced Label Distribution: Counter({0: 6327, 1: 6327})


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.413000,0.695911,0.500198,0.333421,0.250099,0.500000
2,1.261000,0.596251,0.692217,0.688179,0.702802,0.692262
3,1.122400,0.552557,0.724615,0.723695,0.727689,0.724638
4,0.918300,0.511946,0.763730,0.763382,0.765320,0.763745
5,0.897700,0.500642,0.765705,0.759517,0.796346,0.765769
6,0.666500,0.478941,0.799289,0.798453,0.804395,0.799314
7,0.528900,0.478836,0.814698,0.813625,0.822183,0.814728


Final Evaluation Results:
Accuracy: 0.8131
F1-Score: 0.8127
